# Lab — Budgeting: window & cost
<a id="top"></a>

```yaml
title: Lab — Budgeting: window & cost
keywords: context window, headroom, cost, pricing, staircase, budgeting
estimated duration: 25
```

> **Lesson L01 lab.** Objectives 6 & 7 — reason about context windows and estimate cost.
> Fully offline (no API key). Fill each `# TODO`; check against the solutions notebook.

**Contents**

1. [Headroom in the window](#1-headroom-in-the-window)
2. [Match the failure mode](#2-match-the-failure-mode)
3. [Cost of a call](#3-cost-of-a-call)
4. [The conversation staircase](#4-the-conversation-staircase)

In [1]:
from __future__ import annotations

WINDOW = 200_000
# ⚠️ ILLUSTRATIVE $/1M-token rates — replace with current Claude Sonnet pricing before quoting.
INPUT_RATE = 3.0
OUTPUT_RATE = 15.0

## 1. Headroom in the window

Implement `headroom`: tokens left in the window after subtracting the preamble, history, current input, and reserved output. A negative result means overflow.

In [2]:
def headroom(preamble: int, history: int, current: int, reserved: int) -> int:
    return WINDOW - (preamble + history + current + reserved)

In [3]:
print(headroom(preamble=300, history=160_000, current=250, reserved=1_024))

38426


## 2. Match the failure mode

Match each symptom to **hard rejection**, **silent truncation**, or **quality degradation**:

- a) The API returns an error before generating anything. → 
- b) The call succeeds but the model forgot the earliest turns. → 
- c) The call fits, but answers get vaguer about mid-conversation details. → 

## 3. Cost of a call

Implement `call_cost_usd(input_tokens, output_tokens)` using the rates above, then compare 2000-in/50-out vs 50-in/2000-out. Which is more expensive, and why?

In [4]:
def call_cost_usd(input_tokens: int, output_tokens: int) -> float:
    return input_tokens / 1e6 * INPUT_RATE + output_tokens / 1e6 * OUTPUT_RATE


print(f"2000/50  : ${call_cost_usd(2000, 50):.5f}")
print(f"50/2000  : ${call_cost_usd(50, 2000):.5f}  (output costs ~5x more per token)")

2000/50  : $0.00675
50/2000  : $0.03015  (output costs ~5x more per token)


## 4. The conversation staircase

A 5-turn chat adds ~200 tokens/turn, and every turn re-sends all prior tokens. Print the cumulative input re-sent each turn, and the total.

In [5]:
cumulative = 0
for turn in range(1, 6):
    cumulative += 200
    print(f"turn {turn}: cumulative input = {cumulative}")
print("total input re-sent:", sum(range(200, 1200, 200)))

turn 1: cumulative input = 200
turn 2: cumulative input = 400
turn 3: cumulative input = 600
turn 4: cumulative input = 800
turn 5: cumulative input = 1000
total input re-sent: 3000


**Self-check.** `headroom` goes negative on overflow; the short-input/long-output call costs more; the staircase totals 200+400+600+800+1000 = 3000 input tokens.

[↑ Back to top](#top)